In [1]:
import pandas as pd

# Load IMDB dataset
df = pd.read_csv("IMDB Dataset.csv")

# Manually filter based on keywords
action_reviews = df[df['review'].str.contains('fight|war|action|explosion|chase', case=False)].sample(5, random_state=42)
romance_reviews = df[df['review'].str.contains('love|romantic|relationship|kiss|heart', case=False)].sample(5, random_state=42)

# Combine and label
reviews = pd.concat([action_reviews, romance_reviews]).reset_index(drop=True)
labels = ['Action'] * 5 + ['Romance'] * 5
reviews['genre'] = labels


In [2]:
import string
from sklearn.feature_extraction.text import TfidfVectorizer

# Preprocessing
def preprocess(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text

reviews['cleaned'] = reviews['review'].apply(preprocess)

# TF-IDF Vectorization
vectorizer = TfidfVectorizer(stop_words='english')
X = vectorizer.fit_transform(reviews['cleaned'])


In [3]:
from sklearn.metrics.pairwise import cosine_similarity

# Calculate similarity matrix
similarity_matrix = cosine_similarity(X)

# Put in a DataFrame for clarity
import numpy as np
similarity_df = pd.DataFrame(similarity_matrix, index=labels, columns=labels)
print(similarity_df.round(2))


         Action  Action  Action  Action  Action  Romance  Romance  Romance  \
Action     1.00    0.04    0.04    0.06    0.08     0.04     0.06     0.05   
Action     0.04    1.00    0.02    0.02    0.05     0.03     0.02     0.05   
Action     0.04    0.02    1.00    0.05    0.03     0.04     0.01     0.01   
Action     0.06    0.02    0.05    1.00    0.03     0.08     0.05     0.06   
Action     0.08    0.05    0.03    0.03    1.00     0.02     0.04     0.04   
Romance    0.04    0.03    0.04    0.08    0.02     1.00     0.08     0.02   
Romance    0.06    0.02    0.01    0.05    0.04     0.08     1.00     0.01   
Romance    0.05    0.05    0.01    0.06    0.04     0.02     0.01     1.00   
Romance    0.02    0.00    0.00    0.02    0.02     0.02     0.01     0.10   
Romance    0.03    0.03    0.03    0.03    0.02     0.00     0.03     0.05   

         Romance  Romance  
Action      0.02     0.03  
Action      0.00     0.03  
Action      0.00     0.03  
Action      0.02     0.03  
A

In [4]:
import numpy as np

# Average similarities
same_genre_similarities = []
cross_genre_similarities = []

for i in range(10):
    for j in range(10):
        if i != j:
            if labels[i] == labels[j]:
                same_genre_similarities.append(similarity_matrix[i, j])
            else:
                cross_genre_similarities.append(similarity_matrix[i, j])

print("Avg Similarity (Same Genre):", np.mean(same_genre_similarities))
print("Avg Similarity (Different Genre):", np.mean(cross_genre_similarities))


Avg Similarity (Same Genre): 0.03996266341628957
Avg Similarity (Different Genre): 0.033053264992325845
